# MediQ — DistilBERT Symptom Model Fine-Tuning Pipeline

This Google Colab notebook fine-tunes `distilbert-base-uncased` for multi-class symptom-to-condition classification using the **DiseaseSymptom Knowledge Base** dataset (~4,800 records, 41 conditions) augmented with **Kerala-specific endemic conditions** (Nipah, Chikungunya, Leptospirosis) for a total of 44 classes.

### Recommended Runtime
- **Hardware Accelerator**: GPU (T4 or higher)
- Navigate to **Runtime** → **Change runtime type** → select **T4 GPU**.

In [ ]:
# Step 1: Verify GPU Environment
!nvidia-smi

In [ ]:
# Step 2: Install Required Dependencies
!pip install -q transformers datasets accelerate torch scikit-learn pandas matplotlib seaborn

### Step 3: Provide Dataset & Training Scripts

1. Upload the raw dataset file `dataset.csv` (from the DiseaseSymptom Knowledge Base).
2. Upload `kerala_symptoms.json`, `preprocess.py`, `train_symptom_model.py`, and `evaluate.py`.

In [ ]:
from google.colab import files
import os

# Check if dataset exists; if not, prompt file upload
if not os.path.exists('dataset.csv'):
    print('Please upload dataset.csv:')
    uploaded = files.upload()
else:
    print('dataset.csv already present.')

In [ ]:
# Step 4: Run Preprocessing & Stratified Splitting
# Automatically detects dataset columns, integrates Kerala conditions, and outputs train/val/test splits.
!python preprocess.py \
    --dataset_path dataset.csv \
    --kerala_json kerala_symptoms.json \
    --output_dir ./processed \
    --train_ratio 0.80 \
    --val_ratio 0.10 \
    --test_ratio 0.10 \
    --seed 42

In [ ]:
# Inspect preprocessing summary
import json
with open('./processed/preprocessing_summary.json', 'r') as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))

In [ ]:
# Step 5: Fine-Tune DistilBERT on GPU
!python train_symptom_model.py \
    --data_dir ./processed \
    --output_dir ./models/symptom \
    --base_model distilbert-base-uncased \
    --epochs 5 \
    --batch_size 16 \
    --learning_rate 3e-5 \
    --max_seq_len 128 \
    --patience 2

In [ ]:
# Step 6: Evaluate on Held-Out Test Set
!python evaluate.py \
    --model_dir ./models/symptom \
    --data_dir ./processed \
    --output_file ./models/symptom/evaluation_results.json

In [ ]:
# Display Final Test Metrics
with open('./models/symptom/evaluation_results.json', 'r') as f:
    results = json.load(f)

print(f"Accuracy:        {results['accuracy']:.4f}")
print(f"Macro Precision: {results['macro_precision']:.4f}")
print(f"Macro Recall:    {results['macro_recall']:.4f}")
print(f"Macro F1:        {results['macro_f1']:.4f}")

In [ ]:
# Step 7: Local Test Inference Verification
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = './models/symptom'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

with open(f'{model_path}/label_mapping.json') as f:
    id2label = json.load(f)['id2label']

test_phrases = [
    'I have fever, severe joint pain and skin rash',
    'high fever, headache, breathing difficulty and disorientation',
    'fever, calf tenderness and yellowing of eyes after floodwater exposure',
    'cough, sneezing and runny nose',
]

for phrase in test_phrases:
    inputs = tokenizer(phrase, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        topk = torch.topk(probs, k=3)

    print(f'\nQuery: "{phrase}"')
    for score, idx in zip(topk.values, topk.indices):
        disease = id2label[str(idx.item())]
        print(f'  - {disease}: {score.item():.4f}')

In [ ]:
# Step 8: Package Artifacts for Deployment into MediQ Backend
!zip -r mediq_symptom_distilbert.zip ./models/symptom
print('Zipped model artifacts. Downloading...')
files.download('mediq_symptom_distilbert.zip')